# Week 3 — Natural Language Processing & Review Trust Intelligence
## AI-Powered Marketplace Trust Intelligence Platform

### Scope & Architecture
This notebook evaluates customer review text across multiple dimensions to uncover explainable risk signals:
1. **Intra-Listing Review Similarity:** Detects repetitive wording and copy-paste review rings within the same product listing.
2. **Rating–Sentiment Discordance:** Identifies disconnects between numeric star ratings and text sentiment polarity.
3. **Writing Style & Stylometry:** Extracts uppercase density, exclamation bursts, and punctuation repetition.
4. **Multi-Category Generalization:** Validates review signals across Electronics, Beauty, Fashion, Home, Toys, and Books.
5. **Isolated Benchmark Experiment:** Evaluates an external deception benchmark (Ott et al.) strictly in isolation, without conflating hotel reviews with marketplace product risk.


## 1. Load Baseline Processed Dataset


In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent

baseline_path = project_root / 'data' / 'processed' / 'marketplace_text_features.parquet'
df = pd.read_parquet(baseline_path)

print('Dataset shape:', df.shape)
print('Review text missing:', df['review_text'].isna().sum())
print('Unique ratings:', sorted(df['rating'].dropna().unique()))
display(df[['rating', 'review_text']].sample(min(5, len(df)), random_state=42))


## 2. Text Normalization & Global Vocabulary


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create clean NLP text column
df['nlp_text'] = (
    df['review_text']
    .astype('string')
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

nlp_df = df[df['nlp_text'].notna() & df['nlp_text'].str.strip().ne('')].copy()
print('Total usable reviews:', len(nlp_df))
print('Unique normalized reviews:', nlp_df['nlp_text'].nunique())

tfidf = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2), min_df=2)
X_tfidf = tfidf.fit_transform(nlp_df['nlp_text'])
print('TF-IDF matrix shape:', X_tfidf.shape)
print('Vocabulary size:', len(tfidf.vocabulary_))


## 3. Intra-Product Review Similarity Analysis (Dense Cohort)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

dense_reviews_path = project_root / 'data' / 'interim' / 'dense_reviews.parquet'
dense_df = pd.read_parquet(dense_reviews_path)
dense_df['nlp_text'] = (
    dense_df['review_text']
    .astype('string')
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
dense_df = dense_df[dense_df['nlp_text'].notna() & dense_df['nlp_text'].ne('')].copy()

print('Dense cohort reviews:', len(dense_df))
print('Products in dense cohort:', dense_df['parent_asin'].nunique())

similar_pairs = []
for product_id, group in dense_df.groupby('parent_asin'):
    if len(group) < 2:
        continue
    vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=1, max_features=1000)
    X = vec.fit_transform(group['nlp_text'])
    sim_mat = cosine_similarity(X)
    for i in range(len(group)):
        for j in range(i + 1, len(group)):
            score = sim_mat[i, j]
            if score >= 0.80:
                similar_pairs.append({
                    'parent_asin': product_id,
                    'similarity': round(float(score), 3),
                    'review_1': group.iloc[i]['nlp_text'][:80],
                    'review_2': group.iloc[j]['nlp_text'][:80],
                })

similarity_df = pd.DataFrame(similar_pairs)
print(f'Highly similar review pairs (cosine >= 0.80): {len(similarity_df)}')
if not similarity_df.empty:
    display(similarity_df.head(5))


## 4. Sentiment Analysis & Rating–Sentiment Disagreement


In [ ]:
from src.nlp.review_analysis import get_sentiment_compound

dense_df['sentiment_compound'] = dense_df['nlp_text'].apply(get_sentiment_compound)

def rating_category(r):
    return 'Negative' if r <= 2 else ('Neutral' if r == 3 else 'Positive')

def text_sentiment_cat(s):
    return 'Positive' if s >= 0.05 else ('Negative' if s <= -0.05 else 'Neutral')

dense_df['rating_sentiment'] = dense_df['rating'].apply(rating_category)
dense_df['text_sentiment'] = dense_df['sentiment_compound'].apply(text_sentiment_cat)
dense_df['sentiment_disagreement'] = dense_df['rating_sentiment'] != dense_df['text_sentiment']

print('Sentiment disagreement rate:', round(dense_df['sentiment_disagreement'].mean() * 100, 2), '%')
display(pd.crosstab(dense_df['rating_sentiment'], dense_df['text_sentiment'], margins=True))


## 5. Multi-Category Review Analysis (Canonical Platform Dataset)
Evaluates review distributions and sentiment dynamics across multiple categories (Electronics, Fashion, Home, Toys, Books, BestBuy).


In [ ]:
canonical_reviews_path = project_root / 'data' / 'processed' / 'canonical_reviews.parquet'
canonical_prods_path = project_root / 'data' / 'processed' / 'canonical_products.parquet'

df_can_revs = pd.read_parquet(canonical_reviews_path)
df_can_prods = pd.read_parquet(canonical_prods_path)

# Link product categories to reviews
cat_map = dict(zip(df_can_prods['canonical_product_id'], df_can_prods['category']))
df_can_revs['category'] = df_can_revs['canonical_product_id'].map(cat_map).fillna('Other')

print('Canonical reviews by marketplace:')
display(df_can_revs['marketplace'].value_counts())
print('\nCanonical reviews by category:')
display(df_can_revs['category'].value_counts().head(10))


---
## 6. Isolated NLP Deception Benchmark (Ott et al. Corpus)

> **CRITICAL RESEARCH BOUNDARY NOTICE**  
> The following experiment evaluates the Ott et al. (ACL 2011, EMNLP 2013) deceptive opinion spam corpus.  
> **This is a separate research experiment on lexical deception stylometry in hotel reviews.**  
> - It is NOT a marketplace product risk model.  
> - Hotel deception labels are NEVER assigned or transferred to marketplace listings.  
> - Real-world marketplace trust intelligence relies on multi-signal evidence aggregation (Price + Reviews + Images), not hotel sentiment classifiers.  


In [ ]:
benchmark_path = project_root / 'data' / 'raw' / 'benchmarks' / 'deceptive_opinion_spam.parquet'
benchmark_df = pd.read_parquet(benchmark_path)
print('Benchmark dataset shape:', benchmark_df.shape)
display(benchmark_df.head(3))
print('\nClass balance:')
display(benchmark_df['deceptive'].value_counts())


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib

# Grouped split by hotel to strictly prevent entity leakage across folds
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(benchmark_df['text'], benchmark_df['deceptive'], groups=benchmark_df['hotel']))

train_df = benchmark_df.iloc[train_idx]
test_df = benchmark_df.iloc[test_idx]

# Verify strict hotel isolation
assert len(set(train_df['hotel']) & set(test_df['hotel'])) == 0, 'Data leakage detected across hotel groups!'
print('Verified group split:', len(train_df), 'train reviews,', len(test_df), 'test reviews.')

model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=3000, stop_words='english')),
    ('classifier', LogisticRegression(max_iter=500, random_state=42)),
])

model.fit(train_df['text'], train_df['deceptive'])
y_pred = model.predict(test_df['text'])
acc = accuracy_score(test_df['deceptive'], y_pred)
print(f'\nIsolated Benchmark Test Accuracy: {acc:.4f}')
print('\nClassification Report:')
print(classification_report(test_df['deceptive'], y_pred))

# Save model artifact
models_dir = project_root / 'models'
models_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model, models_dir / 'hotel_review_baseline.joblib')
print('Saved baseline benchmark artifact to models/hotel_review_baseline.joblib')
